# 03_异步子 Agent

**来源:** [LangChain Docs — Async Subagents](https://docs.langchain.com/oss/python/deepagents/async-subagents)

本 Notebook 演示 Deep Agents 的异步子 Agent 功能。异步子 Agent 允许监控 Agent 启动后台任务后立即返回，使监控 Agent 在子 Agent 并发工作时继续与用户交互。监控 Agent 可以随时检查进度、发送后续指令或取消任务。

> **注意：** 异步子 Agent 是 Deep Agents 0.5.0 的预览功能，API 可能发生变化。

## 1. 环境准备与导入

In [ ]:
# pip install deepagents langchain langgraph

from deepagents import AsyncSubAgent, create_deep_agent
from dotenv import load_dotenv
load_dotenv()


## 2. 同步子 Agent vs 异步子 Agent

| 维度 | 同步子 Agent | 异步子 Agent |
|------|-------------|-------------|
| 执行模型 | 监控 Agent 等待子 Agent 完成 | 立即返回 Job ID；监控 Agent 继续运行 |
| 并发 | 并行但阻塞 | 并行且非阻塞 |
| 任务中更新 | 不支持 | 通过 `update_async_task` 发送后续指令 |
| 取消 | 不支持 | 通过 `cancel_async_task` 取消正在运行的任务 |
| 状态管理 | 无状态——调用间无持久状态 | 有状态——在独立线程上维护状态 |
| 适用场景 | Agent 需等待结果后再继续 | 可交互管理的长时间复杂任务 |

## 3. 配置异步子 Agent

将异步子 Agent 定义为 `AsyncSubAgent` 规范列表，每个指向一个 Agent Protocol 服务器。

In [ ]:
# 配置异步子 Agent
async_subagents = [
    AsyncSubAgent(
        name="researcher",
        description="用于信息收集和综合的研究 Agent",
        graph_id="researcher",
        # 无 url → ASGI 传输（同部署）
    ),
    AsyncSubAgent(
        name="coder",
        description="用于代码生成和审查的编码 Agent",
        graph_id="coder",
        # url="https://coder-deployment.langsmith.dev"  # 可选：远程 HTTP 传输
    ),
]

# 创建深度 Agent
agent = create_deep_agent(
    model="deepseek-v4-flash",
    subagents=async_subagents,
)

## 4. AsyncSubAgent 配置字段

| 字段 | 类型 | 描述 |
|------|------|------|
| `name` | str | 必填。唯一标识符。监控 Agent 使用此名称启动任务。 |
| `description` | str | 必填。子 Agent 的功能描述。监控 Agent 用于决定委托给哪个 Agent。 |
| `graph_id` | str | 必填。Agent Protocol 服务器上的图 ID（或助手 ID）。对于 LangGraph 部署，必须匹配 `langgraph.json` 中注册的图。 |
| `url` | str | 可选。省略时使用 ASGI 传输（进程内）；设置时使用 HTTP 传输到远程 Agent Protocol 服务器。 |
| `headers` | dict[str, str] | 可选。发送到远程服务器的额外请求头。 |

## 5. langgraph.json 配置

对于同部署设置，所有图需在同一个 `langgraph.json` 中注册：

```json
{
  "graphs": {
    "supervisor": "./src/supervisor.py:graph",
    "researcher": "./src/researcher.py:graph",
    "coder": "./src/coder.py:graph"
  }
}
```

## 6. 异步子 Agent 工具

`AsyncSubAgentMiddleware` 为监控 Agent 提供五种工具：

| 工具 | 用途 | 返回 |
|------|------|------|
| `start_async_task` | 启动新的后台任务 | 立即返回 Task ID |
| `check_async_task` | 获取任务的当前状态和结果 | 状态 + 结果（如已完成） |
| `update_async_task` | 向运行中的任务发送新指令 | 确认 + 更新后的状态 |
| `cancel_async_task` | 停止运行中的任务 | 确认 |
| `list_async_tasks` | 列出所有已跟踪任务及其实时状态 | 所有任务的摘要 |

监控 Agent 的 LLM 像调用其他工具一样调用这些工具。中间件自动处理线程创建、运行管理和状态持久化。

## 7. 生命周期理解

典型交互顺序：

1. **启动（Launch）**：在服务器上创建新线程，以任务描述作为输入启动运行，返回线程 ID 作为任务 ID。监控 Agent 将此 ID 报告给用户，不会轮询完成状态。
2. **检查（Check）**：获取当前运行状态。如果运行成功，检索线程状态以提取子 Agent 的最终输出。如果仍在运行，向用户报告。
3. **更新（Update）**：在同一个线程上以 `interrupt` 多任务策略创建新运行。之前运行被中断，子 Agent 使用完整对话历史加新指令重新开始。任务 ID 保持不变。
4. **取消（Cancel）**：调用 `runs.cancel()` 并将任务标记为 "cancelled"。
5. **列表（List）**：遍历所有已跟踪任务。对于非终止任务，从服务器并行获取实时状态。终止状态（success, error, cancelled）从缓存返回。

## 8. 状态管理

任务元数据存储在监控 Agent 图的专用状态通道（`async_tasks`）中，与消息历史分离。这是关键设计，因为深度 Agent 会在上下文窗口填满时压缩消息历史。如果任务 ID 仅存在于工具消息中，压缩时会被丢失。专用通道确保监控 Agent 始终可以通过 `list_async_tasks` 召回任务，即使在多轮摘要之后。


每个被跟踪的任务记录：任务 ID、Agent 名称、线程 ID、运行 ID、状态和时间戳（`created_at`、`last_checked_at`、`last_updated_at`）。

## 9. 部署选项

### ASGI 传输（同部署）

当子 Agent 规范省略 `url` 字段时，LangGraph SDK 使用 ASGI 传输——SDK 调用通过进程内函数调用路由，而非 HTTP。同部署需要两个图都注册在同一个 `langgraph.json` 中。ASGI 传输消除网络延迟，无需额外认证配置。子 Agent 仍作为独立线程运行，拥有自己的状态。

### HTTP 传输（远程）

添加 `url` 字段切换到 HTTP 传输：

```python
AsyncSubAgent(
    name="researcher",
    description="Research agent",
    graph_id="researcher",
    url="https://my-research-deployment.langsmith.dev",
)
```

对于 LangGraph 部署，认证由 LangGraph SDK 使用环境变量 `LANGSMITH_API_KEY`（或 `LANGGRAPH_API_KEY`）处理。自托管 Agent Protocol 服务器可能使用不同的认证机制。当子 Agent 需要独立扩展、不同资源配置或由不同团队维护时，使用 HTTP 传输。

### 混合部署

在同部署中，一些子 Agent 通过 ASGI 同部署，其他通过 HTTP 远程连接。

In [ ]:
# 混合部署示例：本地 ASGI + 远程 HTTP
async_subagents = [
    AsyncSubAgent(
        name="researcher",
        description="研究 Agent",
        graph_id="researcher",
        # 无 url → ASGI（同部署）
    ),
    AsyncSubAgent(
        name="coder",
        description="编码 Agent",
        graph_id="coder",
        url="https://coder-deployment.langsmith.dev",
        # url 存在 → HTTP（远程）
    ),
]

agent = create_deep_agent(
    model="deepseek-v4-flash",
    subagents=async_subagents,
)

## 10. 本地开发工作池大小

本地使用 `langgraph dev` 运行时，增加工作池以容纳并发子 Agent 运行。每个活动运行占用一个工作槽。一个具有 3 个并发子 Agent 任务的监控 Agent 需要 4 个槽（1 监控 + 3 子 Agent）。配置不足会导致启动排队。

```bash
langgraph dev --n-jobs-per-worker 10
```

## 11. 最佳实践

### 编写清晰的描述

监控 Agent 使用描述来决定启动哪个子 Agent。要具体且以行动为导向：

```python
# 好的做法
AsyncSubAgent(
    name="researcher",
    description="使用网络搜索进行深度研究。适用于需要多次搜索和综合的问题。",
    graph_id="researcher",
)

# 不好的做法
AsyncSubAgent(
    name="helper",
    description="帮忙处理事情",
    graph_id="helper",
)
```

### 使用线程 ID 追踪

每个异步子 Agent 运行都是标准的 LangGraph 运行，可在 LangSmith 中完全查看。监控 Agent 的追踪显示 launch、check、update、cancel 和 list 的工具调用。每个子 Agent 运行显示为单独的追踪，通过线程 ID 关联。

### 常见问题

1. **监控 Agent 启动后立即轮询**：中间件会注入系统提示规则防止。如果仍然轮询，在系统提示中添加规则。
2. **监控 Agent 报告过期状态**：中间件提示指示模型"对话历史中的任务状态始终过期"。
3. **任务 ID 查找失败**：中间件提示指示始终使用完整任务 ID，从不截断或缩写。
4. **子 Agent 启动排队而非运行**：工作池可能已耗尽，增加池大小。

---

**参考链接**
- [Deep Agents — Async Subagents](https://docs.langchain.com/oss/python/deepagents/async-subagents)
- [子 Agent](https://docs.langchain.com/oss/python/deepagents/subagents)
- [Agent Protocol](https://agentclientprotocol.com/)
- [LangSmith 部署](https://docs.smith.langchain.com/)
- [异步深度 Agent 参考实现](https://github.com/langchain-ai/async-deep-agents)